# Workshop 2.4: Measuring Strategy Performance

Welcome to Workshop 2.4! In Workshop 2.3, we engineered our first moving average crossover strategy. But how do we truly determine if an algorithmic strategy is successful?

### Beyond Raw Headline Returns

A strategy generating thirty percent return might look appealing on the surface, but if that return required enduring a seventy percent peak-to-trough collapse along the way, most investors would abandon it in panic. In professional quantitative finance, returns must always be weighed against the risks taken to earn them.

To evaluate strategies with institutional rigor, we rely on a dashboard of **risk-adjusted metrics**:
- **Annualized Return**: The compound annual growth rate of our capital.
- **Sharpe Ratio**: The excess return generated per unit of volatility.
- **Maximum Drawdown**: The deepest peak-to-trough loss, measuring capital preservation.
- **Win Rate and Win/Loss Ratio**: The frequency and magnitude of profitable sessions.

In this workshop, we will construct each of these core performance benchmarks and compile them into a unified evaluation card.

> **Key Takeaway**: Evaluating algorithmic trading strategies requires balancing raw returns against drawdowns and risk-adjusted efficiency.

## Topic 1: Cumulative and Annualized Return

The most direct measure of performance is **total return**, representing the overall percentage change in your capital from start to finish.

Because different backtests span different time periods, we also compute the **annualized compound return** to compare strategies across standardized annual horizons.

Let's reconstruct our SMA strategy from Workshop 2.3 and compute both return metrics. Let's see:

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load AAPL 2023 dataset:
try:
    aapl = pd.read_parquet("aapl_2023.parquet")
except Exception:
    import yfinance as yf
    aapl = yf.download("AAPL", start="2023-01-01", end="2024-01-01", progress=False)
    aapl.index = aapl.index.tz_localize(None)

# Calculate returns and moving averages:
aapl["Return"] = aapl["Close"].pct_change()
aapl["SMA_20"] = aapl["Close"].rolling(20).mean()
aapl["SMA_50"] = aapl["Close"].rolling(50).mean()

# Generate signal and lagged position:
aapl["Signal"] = 0
aapl.loc[aapl["SMA_20"] > aapl["SMA_50"], "Signal"] = 1
aapl.loc[aapl["SMA_20"] < aapl["SMA_50"], "Signal"] = -1
aapl["Position"] = aapl["Signal"].shift(1)
aapl["Strategy_Return"] = aapl["Position"] * aapl["Return"]

# We calculate cumulative portfolio equity:
aapl["Cumulative_Strategy"] = (1 + aapl["Strategy_Return"]).cumprod()
total_return = aapl["Cumulative_Strategy"].iloc[-1] - 1

# We annualize returns assuming 252 trading days:
n_days = len(aapl)
annualized_return = (1 + total_return) ** (252 / n_days) - 1

print(f"Total Return:      {total_return:.4f} ({total_return * 100:.2f}%)")
print(f"Annualized Return: {annualized_return:.4f} ({annualized_return * 100:.2f}%)")

Total Return:      10.43%
Annualized Return: 10.52%


> **Key Takeaway**: Total return tracks overall wealth growth, while annualized return scales performance to a standard 252-day calendar year.

---

## Topic 2: The Sharpe Ratio: Rewarding Efficiency

Introduced by Nobel laureate William Sharpe, the **Sharpe Ratio** answers a vital question: *"How much return did the strategy generate per unit of volatility risk?"*

Think of the Sharpe Ratio like a car's fuel efficiency rating: it measures how many miles of return you travel per gallon of volatility stress. Higher values indicate smoother, more dependable performance.

Assuming a zero risk-free rate for daily simplicity, the formula is:
$$\text{Sharpe Ratio} = \frac{\text{Mean Strategy Return}}{\text{Std Dev of Strategy Return}} \times \sqrt{252}$$

Let's compute the Sharpe Ratio for our trend strategy. Let's check:

In [2]:
# Calculate daily mean and daily volatility:
daily_mean = aapl["Strategy_Return"].mean()
daily_std = aapl["Strategy_Return"].std()

# Annualize the Sharpe Ratio:
sharpe_ratio = (daily_mean / daily_std) * np.sqrt(252)

print(f"Daily Mean Return:     {daily_mean:.6f}")
print(f"Daily Volatility (std): {daily_std:.6f}")
print(f"Annualized Sharpe Ratio: {sharpe_ratio:.4f}")

Mean Daily Return: 0.000454
Std Daily Return:  0.010607
Sharpe Ratio:      0.68


> **Key Takeaway**: The Sharpe Ratio measures risk-adjusted return by dividing annualized excess return by annualized volatility.

---

## Topic 3: Maximum Drawdown: Gauging Capital Pain

Even profitable strategies experience painful losing stretches. **Maximum Drawdown** measures the largest percentage decline from an equity peak to a subsequent trough.

Think of maximum drawdown like descending into an underwater canyon between mountain summits:
- **High-Water Mark (Peak)**: The highest portfolio value achieved up to that date, tracked using `.cummax()`.
- **Drawdown**: The percentage dip below that high-water mark.
- **Maximum Drawdown**: The deepest valley encountered across the entire trading timeline.

Let's calculate the high-water mark and drawdown series. Let's see:

In [3]:
# We track the running peak or high-water mark:
peak = aapl["Cumulative_Strategy"].cummax()

# We measure the percentage drop from that peak:
drawdown = (aapl["Cumulative_Strategy"] - peak) / peak

# Maximum drawdown is the deepest valley:
max_drawdown = drawdown.min()

print(f"Maximum Drawdown: {max_drawdown:.4f} ({max_drawdown * 100:.2f}%)")

Maximum Drawdown: -16.56%


> **Key Takeaway**: Maximum drawdown identifies the worst peak-to-trough decline, measuring psychological and financial stress during market downturns.

---

## Topic 4: Win Rate and Average Win/Loss

A strategy does not need a high win rate to be profitable. Many successful trend followers win only forty percent of their trades, but their average winning session significantly outpaces their average loss.

We calculate:
- **Win Rate**: The proportion of active sessions producing positive returns.
- **Average Win vs Loss**: The average gain on positive days compared against average losses on negative days.

Let's calculate these trade-level statistics. Let's check:

In [4]:
# Separate winning and losing days:
wins = aapl[aapl["Strategy_Return"] > 0]
losses = aapl[aapl["Strategy_Return"] < 0]

# Calculate win rate and average win / loss:
win_rate = len(wins) / (len(wins) + len(losses))
avg_win = wins["Strategy_Return"].mean()
avg_loss = losses["Strategy_Return"].mean()

# Print all values:
print(f"Number of Wins:   {len(wins)}")
print(f"Number of Losses: {len(losses)}")
print(f"Win Rate:         {win_rate:.2%}")
print(f"Average Win:      {avg_win:.2%}")
print(f"Average Loss:     {avg_loss:.2%}")

Number of Wins:   93
Number of Losses: 106
Win Rate:         46.73%
Average Win:      1.02%
Average Loss:     -0.79%


> **Key Takeaway**: Profitability depends on the balance between win rate and the ratio of average win to average loss.

---

## Topic 5: Compiling a Strategy Scorecard

Combining all key metrics into a single summary table provides an executive dashboard for comparing different algorithms side by side.

Let's build our summary table. Let's see:

In [5]:
# Create performance summary table:
performance_summary = pd.DataFrame({
    "Metric": ["Total Return", "Annualized Return", "Sharpe Ratio", "Max Drawdown", "Win Rate"],
    "Value": [f"{total_return:.2%}", f"{annualized_return:.2%}", f"{sharpe_ratio:.2f}", f"{max_drawdown:.2%}", f"{win_rate:.2%}"]
})
print(performance_summary)

              Metric    Value
0       Total Return   10.43%
1  Annualized Return   10.52%
2       Sharpe Ratio     0.68
3       Max Drawdown  -16.56%
4           Win Rate   46.73%


> **Key Takeaway**: A unified scorecard aggregates profitability alongside risk metrics into a clear evaluation summary.

---

## Topic 6: Visualizing the Underwater Drawdown Curve

An underwater plot charts portfolio drawdowns over time, revealing how long recovery periods lasted and when capital was exposed to loss.

Let's visualize our strategy's drawdown profile. Let's check:

In [6]:
# Visualize Drawdown over time:
print("Displaying Strategy Drawdown chart:")
plt.figure(figsize=(12, 6))
plt.plot(drawdown, color="red", label="Drawdown")
plt.axhline(y=0, color="black", linestyle="-", linewidth=0.5)
plt.fill_between(drawdown.index, 0, drawdown, color="red", alpha=0.3)
plt.title("Strategy Drawdown Over Time")
plt.ylabel("Drawdown")
plt.legend()
plt.show()

Displaying Strategy Drawdown chart:


> **Key Takeaway**: Underwater plots expose the severity and recovery duration of capital drawdowns across market cycles.

---

## Practice Time

Now it is your turn to calculate and interpret performance metrics across alternative strategy configurations. Evaluating strategies honestly separates solid trading ideas from statistical noise.

---

### Challenge 1: Evaluating a Faster SMA 10/30 Model

- Recompute performance metrics for the `SMA_10` / `SMA_30` configuration from Workshop 2.3.
- Compare total return, Sharpe ratio, and maximum drawdown against our 20/50 baseline.

In [ ]:
# Challenge 1: Re-calculate performance metrics for the SMA_10 / SMA_30 strategy
# Write your code below this line:




### Challenge 2: Benchmarking the Sharpe Ratio Against Buy-and-Hold

- Calculate the annualized Sharpe ratio for the passive buy-and-hold strategy.
- Compare the result against our moving average strategy.

In [ ]:
# Challenge 2: Calculate the Sharpe ratio for the buy-and-hold strategy
# Write your code below this line:




### Challenge 3: Interpreting Industry Sharpe Thresholds

- Research how institutional hedge funds evaluate Sharpe ratios.
- Summarize typical benchmark ranges for quantitative trading strategies.

In [ ]:
# Challenge 3: Research what a good Sharpe ratio is. Write your summary below:




---

## Solutions Section

Great work completing these performance evaluation challenges! Learning to diagnose strategy health through risk-adjusted metrics is what makes a professional quantitative developer.

Let's review the reference implementations together.

### Reference Code

#### Solution for Challenge 1:
```python
aapl["SMA_10"] = aapl["Close"].rolling(10).mean()
aapl["SMA_30"] = aapl["Close"].rolling(30).mean()

aapl["Signal_10_30"] = 0
aapl.loc[aapl["SMA_10"] > aapl["SMA_30"], "Signal_10_30"] = 1
aapl.loc[aapl["SMA_10"] < aapl["SMA_30"], "Signal_10_30"] = -1

aapl["Position_10_30"] = aapl["Signal_10_30"].shift(1)
aapl["Strategy_Return_10_30"] = aapl["Position_10_30"] * aapl["Return"]
aapl["Cumulative_10_30"] = (1 + aapl["Strategy_Return_10_30"]).cumprod()

tot_10_30 = aapl["Cumulative_10_30"].iloc[-1] - 1
ann_10_30 = (1 + tot_10_30) ** (252 / len(aapl)) - 1
mean_10_30 = aapl["Strategy_Return_10_30"].mean()
std_10_30 = aapl["Strategy_Return_10_30"].std()
sharpe_10_30 = (mean_10_30 / std_10_30) * np.sqrt(252)

peak_10_30 = aapl["Cumulative_10_30"].cummax()
dd_10_30 = (aapl["Cumulative_10_30"] - peak_10_30) / peak_10_30
max_dd_10_30 = dd_10_30.min()

wins_10_30 = aapl[aapl["Strategy_Return_10_30"] > 0]
losses_10_30 = aapl[aapl["Strategy_Return_10_30"] < 0]
win_rate_10_30 = len(wins_10_30) / (len(wins_10_30) + len(losses_10_30))

print(f"SMA 10/30 Total Return:      {tot_10_30:.2%}")
print(f"SMA 10/30 Annualized Return: {ann_10_30:.2%}")
print(f"SMA 10/30 Sharpe Ratio:      {sharpe_10_30:.2f}")
print(f"SMA 10/30 Max Drawdown:      {max_dd_10_30:.2%}")
print(f"SMA 10/30 Win Rate:          {win_rate_10_30:.2%}")
```

#### Solution for Challenge 2:
```python
bh_mean = aapl["Return"].mean()
bh_std = aapl["Return"].std()
bh_sharpe = (bh_mean / bh_std) * np.sqrt(252)

print(f"Buy and Hold Sharpe Ratio: {bh_sharpe:.2f}")
print(f"SMA 20/50 Sharpe Ratio:    {sharpe_ratio:.2f}")
```

#### Solution for Challenge 3:
```python
summary = (
    "Industry Sharpe Ratio Guidelines:\n"
    "- Below 1.0: Sub-optimal for active strategies.\n"
    "- 1.0 to 1.99: Good, typical of successful quant funds.\n"
    "- 2.0 to 2.99: Excellent risk-adjusted returns.\n"
    "- 3.0 or higher: Outstanding (always check for look-ahead bias)."
)
print(summary)
```

---

### Running the Solutions

Let's run each solution cell to verify our expected outputs:

In [7]:
# Solution for Challenge 1:
aapl["SMA_10"] = aapl["Close"].rolling(10).mean()
aapl["SMA_30"] = aapl["Close"].rolling(30).mean()

aapl["Signal_10_30"] = 0
aapl.loc[aapl["SMA_10"] > aapl["SMA_30"], "Signal_10_30"] = 1
aapl.loc[aapl["SMA_10"] < aapl["SMA_30"], "Signal_10_30"] = -1

aapl["Position_10_30"] = aapl["Signal_10_30"].shift(1)
aapl["Strategy_Return_10_30"] = aapl["Position_10_30"] * aapl["Return"]
aapl["Cumulative_10_30"] = (1 + aapl["Strategy_Return_10_30"]).cumprod()

tot_10_30 = aapl["Cumulative_10_30"].iloc[-1] - 1
ann_10_30 = (1 + tot_10_30) ** (252 / len(aapl)) - 1
mean_10_30 = aapl["Strategy_Return_10_30"].mean()
std_10_30 = aapl["Strategy_Return_10_30"].std()
sharpe_10_30 = (mean_10_30 / std_10_30) * np.sqrt(252)

peak_10_30 = aapl["Cumulative_10_30"].cummax()
dd_10_30 = (aapl["Cumulative_10_30"] - peak_10_30) / peak_10_30
max_dd_10_30 = dd_10_30.min()

wins_10_30 = aapl[aapl["Strategy_Return_10_30"] > 0]
losses_10_30 = aapl[aapl["Strategy_Return_10_30"] < 0]
win_rate_10_30 = len(wins_10_30) / (len(wins_10_30) + len(losses_10_30))

print(f"SMA 10/30 Total Return:      {tot_10_30:.2%}")
print(f"SMA 10/30 Annualized Return: {ann_10_30:.2%}")
print(f"SMA 10/30 Sharpe Ratio:      {sharpe_10_30:.2f}")
print(f"SMA 10/30 Max Drawdown:      {max_dd_10_30:.2%}")
print(f"SMA 10/30 Win Rate:          {win_rate_10_30:.2%}")

SMA 10/30 Total Return:      -2.68%
SMA 10/30 Annualized Return: -2.70%
SMA 10/30 Sharpe Ratio:      -0.06
SMA 10/30 Max Drawdown:      -26.30%
SMA 10/30 Win Rate:          47.03%


In [8]:
# Solution for Challenge 2:
bh_mean = aapl["Return"].mean()
bh_std = aapl["Return"].std()
bh_sharpe = (bh_mean / bh_std) * np.sqrt(252)

print(f"Buy and Hold Sharpe Ratio: {bh_sharpe:.2f}")
print(f"SMA 20/50 Sharpe Ratio:    {sharpe_ratio:.2f}")

Buy and Hold Sharpe Ratio: 2.29
SMA 20/50 Sharpe Ratio:    0.68


In [9]:
# Solution for Challenge 3:
summary = (
    "Industry Sharpe Ratio Guidelines:\n"
    "- Below 1.0: Sub-optimal for active strategies.\n"
    "- 1.0 to 1.99: Good, typical of successful quant funds.\n"
    "- 2.0 to 2.99: Excellent risk-adjusted returns.\n"
    "- 3.0 or higher: Outstanding (always check for look-ahead bias)."
)
print(summary)

Industry Sharpe Ratio Guidelines:
- Below 1.0: Sub-optimal for active strategies.
- 1.0 to 1.99: Good, typical of successful quant funds.
- 2.0 to 2.99: Excellent risk-adjusted returns.
- 3.0 or higher: Outstanding (always check for look-ahead bias).
